# Extract GFv1.1 Geofabric from HyTest OSN

Download the GFv1.1 HRU fabric (nhru_v1_1) from the HyTest S3 catalog
and save locally as a GeoPackage for use with `nhf-targets init`.

Source: [HyTest intake catalog](https://github.com/hytest-org/hytest/blob/main/dataset_catalog/hytest_intake_catalog.yml)
DOI: https://doi.org/10.5066/P971JAGF

In [ ]:
import geopandas as gpd
import urllib.request
from pathlib import Path


In [ ]:
# HyTest OSN endpoint and GFv1.1 layers
BASE_URL = "https://usgs.osn.mghpcc.org/hytest/geofabric_v1_1"

LAYERS = {
    "nhru_v1_1": "GFv1.1_nhru_v1_1.geoparquet",
    "nsegment_v1_1": "GFv1.1_nsegment_v1_1.geoparquet",
    "POIs_v1_1": "GFv1.1_POIs_v1_1.geoparquet",
}

# Output directory — change as needed
OUT_DIR = Path("../data/geofabric")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Download each layer, then read locally
# (direct HTTP streaming triggers a snappy decompression bug in pyarrow)
gdfs = {}
for name, filename in LAYERS.items():
    url = f"{BASE_URL}/{filename}"
    local = OUT_DIR / filename
    if not local.exists():
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(url, local)
        print(f"  saved to {local} ({local.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"{name} already downloaded ({local.stat().st_size / 1e6:.1f} MB)")
    gdfs[name] = gpd.read_parquet(local)
    print(f"  {gdfs[name].shape[0]:,} features, CRS={gdfs[name].crs}")


In [ ]:
# Inspect the HRU layer
nhru = gdfs["nhru_v1_1"]
print(f"Columns: {nhru.columns.tolist()}")
print(f"CRS: {nhru.crs}")
print(f"Bounds: {nhru.total_bounds}")
nhru.head()


In [ ]:
# Save each layer as a GeoPackage
for name, gdf in gdfs.items():
    out_path = OUT_DIR / f"{name}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"Saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
nhru.to_crs(4326).total_bounds

